In [108]:
import sys
import pandas as pd
import numpy as np
import torch

# from sklearn.metrics import mean_squared_error, mean_absolute_error
# from transformers import Trainer, TrainingArguments
# from transformers import PatchTSTConfig, PatchTSTForPrediction

from transformers import (
    EarlyStoppingCallback,
    PatchTSTConfig,
    PatchTSTForPrediction,
    Trainer,
    TrainingArguments    
)

import matplotlib.pyplot as plt

sys.path.append('../src')

from dataset import RepositorioDados
from models.har import HarModel

# Detecta o dispositivo e a precisão usada nas operações
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.bfloat16 if (device.type == "cuda" and torch.cuda.is_bf16_supported()) else torch.float32
print(f"Device: {device} | Precision: {dtype}")

# Seed para resultados reproduzíveis
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

Device: cpu | Precision: torch.float32


In [109]:
# ==================== CONFIGURAÇÕES PRÉ-TREINAMENTO ====================
TIMESTAMP_COLUMN = 'timestamp'  # Coluna com timestamps
TARGET_COLUMN = ['Vol']     # Coluna a prever (volatilidade)
FEATURES = ["Vol_lag_1", "Vol_week_mean", "Vol_month_mean"]
ID_COLUMNS = []             # Sem IDs (série única)

# Tamanhos das janelas: histórico 7 dias, previsão 1 dia (dados diários)
CONTEXT_LENGTH = 512         # Janela histórica: x dias passados
FORECAST_HORIZON = 1        # Prever: x dias à frente

TRAIN_FRAC, VALID_FRAC = 0.7, 0.1  # Frações treino/validação/teste

# Hyperparâmetros do modelo
PATCH_LENGTH = 1            # Tamanho do patch (1=sem patchificação, mantém cada dia)
BATCH_SIZE = 32             # Samples por batch (reduzir se GPU memory limitada)
NUM_WORKERS = 0             # Workers para data loading (0 em Windows)
EPOCHS = 50                 # Reduzido: 50→30 (volatilidade tem ciclos curtos)
LEARNING_RATE = 1e-4        # Taxa de aprendizado

In [110]:
# ==================== MÉTRICAS DE AVALIAÇÃO ====================
from sklearn.metrics import mean_squared_error, mean_absolute_error

def compute_metrics(eval_pred):
    pred, labels = eval_pred
    pred = pred[0] if isinstance(pred, tuple) else pred

    # 1-step ahead
    pred_1 = pred[:, 0, 0]
    label_1 = labels[:, 0, 0]

    mse = mean_squared_error(label_1, pred_1)
    mae = mean_absolute_error(label_1, pred_1)
    rmse = np.sqrt(mse)
    mape = np.mean(
        np.abs((label_1 - pred_1) / (np.abs(label_1) + 1e-9))
    ) * 100

    return {
        "MSE": mse,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape
    }

In [111]:
import matplotlib.pyplot as plt
def evaluate_and_visualize(
    model,
    test_dataset,
    tsp,
    test_df,
    model_name="Modelo"
):
    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir="metric_temp",
            per_device_eval_batch_size=32,
            label_names=["future_values"]
        )
    )

    outputs = trainer.predict(test_dataset)

    preds = outputs.predictions[0] if isinstance(outputs.predictions, tuple) else outputs.predictions
    labels = outputs.label_ids

    # Desnormalização
    C = len(tsp.target_columns)
    scaler = next(iter(tsp.target_scaler_dict.values()))
    
    pred_original = scaler.inverse_transform(preds.reshape(-1, C)).reshape(preds.shape)
    labels_original = scaler.inverse_transform(labels.reshape(-1, C)).reshape(labels.shape)

    # 1-step ahead
    preds_1step = pred_original[:, 0, 0]
    labels_1step = labels_original[:, 0, 0]

    # Datas corretas
    test_dates = test_df[TIMESTAMP_COLUMN].values
    forecast_dates = test_dates[CONTEXT_LENGTH : CONTEXT_LENGTH + len(preds_1step)]

    # Métricas finais (fora do Trainer)
    metrics = {
        "MSE": mean_squared_error(labels_1step, preds_1step),
        "MAE": mean_absolute_error(labels_1step, preds_1step),
        "RMSE": np.sqrt(mean_squared_error(labels_1step, preds_1step)),
        "MAPE": np.mean(np.abs((labels_1step - preds_1step) / (np.abs(labels_1step) + 1e-9))) * 100
    }

    print(f"=== AVALIAÇÃO FINAL – {model_name} ===\n")
    for k, v in metrics.items():
        print(f"{k}: {v:.6e}")

    plt.figure(figsize=(12, 6))
    plt.plot(forecast_dates, labels_1step, label="True", color="blue")
    plt.plot(forecast_dates, preds_1step, label="Predicted", linestyle="--", color="red")
    plt.title(f"{model_name} – True vs Predicted Volatility")
    plt.legend()
    plt.grid()
    plt.tight_layout()
    plt.show()

    return forecast_dates, metrics

In [112]:
repo = RepositorioDados()

# Grid Search - Configuração

Vamos definir os espaços de busca para:
1. **Hiperparâmetros do modelo** (d_model, attention heads, layers, dropout)
2. **Parâmetros de treinamento** (learning rate, batch size)
3. **Features dos dados** (diferentes combinações de features)
4. **Janelas temporais** (context_length)

In [113]:
from itertools import product
import json
from datetime import datetime

# ==================== GRID SEARCH - ESPAÇOS DE BUSCA ====================

# 1. Features - Teste diferentes combinações
FEATURE_COMBINATIONS = [
    ["Vol_lag_1", "Vol_week_mean", "Vol_month_mean"],
    ["Vol_lag_1", "Vol_week_mean"],
    ["Vol_lag_1"],
    ["Vol_lag_1", "Vol_lag_2", "Vol_lag_3"],
]

# 2. Janelas temporais
CONTEXT_LENGTHS = [256, 512, 768]
FORECAST_HORIZONS = [1]  # Manter fixo por enquanto

# 3. Hiperparâmetros do modelo
MODEL_PARAMS = {
    'd_model': [64, 128],
    'num_attention_heads': [8, 16],
    'num_hidden_layers': [2, 3],
    'ffn_dim': [256, 512],
    'dropout': [0.1, 0.2],
    'patch_length': [1, 16],
}

# 4. Hiperparâmetros de treinamento
TRAINING_PARAMS = {
    'learning_rate': [1e-4, 5e-4],
    'batch_size': [32, 64],
}

# Configurações fixas
FIXED_PARAMS = {
    'epochs': 30,  # Reduzido para grid search
    'early_stopping_patience': 5,
    'num_workers': 0,
    'train_frac': 0.7,
    'valid_frac': 0.1,
}

print("✓ Espaços de busca definidos")
print(f"Total de combinações de features: {len(FEATURE_COMBINATIONS)}")
print(f"Total de combinações de context_length: {len(CONTEXT_LENGTHS)}")
print(f"Total de combinações de modelo: {np.prod([len(v) for v in MODEL_PARAMS.values()])}")
print(f"Total de combinações de treinamento: {np.prod([len(v) for v in TRAINING_PARAMS.values()])}")
total = len(FEATURE_COMBINATIONS) * len(CONTEXT_LENGTHS) * np.prod([len(v) for v in MODEL_PARAMS.values()]) * np.prod([len(v) for v in TRAINING_PARAMS.values()])
print(f"\n🔍 Total de experimentos: {int(total)}")

✓ Espaços de busca definidos
Total de combinações de features: 4
Total de combinações de context_length: 3
Total de combinações de modelo: 64
Total de combinações de treinamento: 4

🔍 Total de experimentos: 3072


# 🚀 Otimizações para Grid Search Rápido

Estratégias para acelerar sem perder qualidade:
1. **Random Search** - Testar amostra aleatória ao invés de todas combinações
2. **Epochs reduzidas** - 5-10 epochs na busca, retreinar o melhor com mais epochs
3. **Early Stopping agressivo** - Patience reduzido (2-3 epochs)
4. **Subset de dados** - Usar 50-70% dos dados durante busca
5. **Busca Bayesiana (Optuna)** - Algoritmo inteligente que aprende quais parâmetros testar

In [114]:
# ==================== CONFIGURAÇÕES DE OTIMIZAÇÃO ====================

# Estratégia 1: Random Search (mais rápido que Grid Search completo)
USE_RANDOM_SEARCH = True  # True = Random Search | False = Grid Search completo
N_RANDOM_TRIALS = 50      # Número de combinações aleatórias a testar

# Estratégia 2: Reduzir epochs durante busca (retreinar o melhor com mais epochs depois)
SEARCH_EPOCHS = 10        # Epochs durante busca (reduzido!)
FINAL_EPOCHS = 50         # Epochs para retreinar o melhor modelo

# Estratégia 3: Early stopping mais agressivo
SEARCH_PATIENCE = 3       # Patience durante busca (mais agressivo)
FINAL_PATIENCE = 5        # Patience no modelo final

# Estratégia 4: Usar subset dos dados (opcional)
USE_DATA_SUBSET = False   # True = usar apenas parte dos dados na busca
SUBSET_FRACTION = 0.5     # Fração dos dados a usar (50%)

# Estratégia 5: Espaços de busca reduzidos (mais inteligentes)
FEATURE_COMBINATIONS_FAST = [
    ["Vol_lag_1", "Vol_week_mean", "Vol_month_mean"],
    ["Vol_lag_1", "Vol_week_mean"],
    ["Vol_lag_1"],
]

CONTEXT_LENGTHS_FAST = [256, 512]  # Removido 768 para acelerar

MODEL_PARAMS_FAST = {
    'd_model': [64, 128],                    # Mantido
    'num_attention_heads': [8, 16],          # Mantido
    'num_hidden_layers': [2, 3],             # Mantido
    'ffn_dim': [256, 512],                   # Mantido
    'dropout': [0.1, 0.2],                   # Mantido
    'patch_length': [1, 16],                 # Mantido
}

TRAINING_PARAMS_FAST = {
    'learning_rate': [1e-4, 5e-4],           # Mantido
    'batch_size': [32, 64],                  # Mantido
}

# Atualizar FIXED_PARAMS com valores otimizados
FIXED_PARAMS_FAST = {
    'epochs': SEARCH_EPOCHS,
    'early_stopping_patience': SEARCH_PATIENCE,
    'num_workers': 0,
    'train_frac': 0.7 if not USE_DATA_SUBSET else 0.5,  # Reduzir se subset
    'valid_frac': 0.1 if not USE_DATA_SUBSET else 0.1,
}

print("⚡ CONFIGURAÇÃO DE BUSCA RÁPIDA")
print("="*80)
print(f"Estratégia: {'RANDOM SEARCH' if USE_RANDOM_SEARCH else 'GRID SEARCH COMPLETO'}")
if USE_RANDOM_SEARCH:
    print(f"Tentativas: {N_RANDOM_TRIALS}")
print(f"Epochs por experimento: {SEARCH_EPOCHS} (final: {FINAL_EPOCHS})")
print(f"Early stopping patience: {SEARCH_PATIENCE} (final: {FINAL_PATIENCE})")
print(f"Usar subset de dados: {'SIM - ' + str(int(SUBSET_FRACTION*100)) + '%' if USE_DATA_SUBSET else 'NÃO - 100%'}")
print("="*80)

# Calcular total de combinações
total_combinations = (
    len(FEATURE_COMBINATIONS_FAST) * 
    len(CONTEXT_LENGTHS_FAST) * 
    np.prod([len(v) for v in MODEL_PARAMS_FAST.values()]) * 
    np.prod([len(v) for v in TRAINING_PARAMS_FAST.values()])
)

if USE_RANDOM_SEARCH:
    actual_trials = min(N_RANDOM_TRIALS, total_combinations)
    print(f"🔍 Testando {actual_trials} de {total_combinations} combinações possíveis")
    print(f"⏱️ Tempo estimado: {actual_trials * 1 / 60:.1f} horas (~1 min/experimento)")
else:
    print(f"🔍 Testando TODAS as {total_combinations} combinações")
    print(f"⏱️ Tempo estimado: {total_combinations * 2 / 60:.1f} horas (~2 min/experimento)")
print("="*80)

⚡ CONFIGURAÇÃO DE BUSCA RÁPIDA
Estratégia: RANDOM SEARCH
Tentativas: 50
Epochs por experimento: 10 (final: 50)
Early stopping patience: 3 (final: 5)
Usar subset de dados: NÃO - 100%
🔍 Testando 50 de 1536 combinações possíveis
⏱️ Tempo estimado: 0.8 horas (~1 min/experimento)


# 🎯 OPÇÃO AVANÇADA: Busca Bayesiana com Optuna

**Optuna** é mais inteligente que Random/Grid Search:
- Aprende com experimentos anteriores
- Foca em regiões promissoras do espaço de busca
- Pode encontrar melhores configurações com menos tentativas
- **Recomendado para espaços de busca grandes**

Para usar Optuna: `pip install optuna`

In [115]:
# Instalar Optuna (descomente se necessário)
# !pip install optuna

try:
    import optuna
    from optuna.samplers import TPESampler
    OPTUNA_AVAILABLE = True
    print("✅ Optuna disponível - Busca Bayesiana ativada")
except ImportError:
    OPTUNA_AVAILABLE = False
    print("⚠️ Optuna não instalado - Use: pip install optuna")
    print("   Busca padrão (Random/Grid) será usada")

⚠️ Optuna não instalado - Use: pip install optuna
   Busca padrão (Random/Grid) será usada


In [116]:
def optuna_objective(trial):
    """
    Função objetivo para Optuna.
    Retorna a métrica a ser MINIMIZADA (eval_loss ou RMSE).
    """
    # Sugerir hiperparâmetros
    features_idx = trial.suggest_categorical('features_idx', [0, 1, 2])
    feature_map = {
        0: ["Vol_lag_1", "Vol_week_mean", "Vol_month_mean"],
        1: ["Vol_lag_1", "Vol_week_mean"],
        2: ["Vol_lag_1"]
    }
    features = feature_map[features_idx]
    
    context_length = trial.suggest_categorical('context_length', [256, 512])
    d_model = trial.suggest_categorical('d_model', [64, 128])
    num_attention_heads = trial.suggest_categorical('num_attention_heads', [8, 16])
    num_hidden_layers = trial.suggest_int('num_hidden_layers', 2, 3)
    ffn_dim = trial.suggest_categorical('ffn_dim', [256, 512])
    dropout = trial.suggest_float('dropout', 0.1, 0.2, step=0.1)
    patch_length = trial.suggest_categorical('patch_length', [1, 16])
    learning_rate = trial.suggest_categorical('learning_rate', [1e-4, 5e-4])
    batch_size = trial.suggest_categorical('batch_size', [32, 64])
    
    # Executar experimento
    result = run_experiment(
        features=features,
        context_length=context_length,
        forecast_horizon=1,
        d_model=d_model,
        num_attention_heads=num_attention_heads,
        num_hidden_layers=num_hidden_layers,
        ffn_dim=ffn_dim,
        dropout=dropout,
        patch_length=patch_length,
        learning_rate=learning_rate,
        batch_size=batch_size,
        experiment_id=trial.number,
        use_fast_config=True
    )
    
    # Salvar resultado completo como atributo do trial
    trial.set_user_attr('full_result', result)
    
    # Se falhou, retornar valor alto
    if result['status'] == 'failed':
        return float('inf')
    
    # Retornar métrica a minimizar
    return result['eval_RMSE']  # ou 'eval_loss'

if OPTUNA_AVAILABLE:
    print("✓ Função objetivo Optuna definida")
else:
    print("⚠️ Optuna não disponível - função objetivo não será usada")

⚠️ Optuna não disponível - função objetivo não será usada


# 🚀 Executar Busca Bayesiana com Optuna (ALTERNATIVA)

**Use esta célula SE quiser usar Optuna ao invés de Random/Grid Search**
- Mais inteligente e eficiente
- Melhor para espaços de busca grandes
- Requer `pip install optuna`

In [117]:
# ============ EXECUTAR BUSCA COM OPTUNA (ALTERNATIVA) ============
# Descomente e execute esta célula para usar Optuna ao invés do grid/random search

"""
if OPTUNA_AVAILABLE:
    # Configurar estudo Optuna
    study = optuna.create_study(
        direction='minimize',  # Minimizar RMSE
        sampler=TPESampler(seed=RANDOM_STATE),
        study_name='patchtst_optimization'
    )
    
    print(f"🎯 Iniciando busca Bayesiana com Optuna")
    print(f"Tentativas: {N_RANDOM_TRIALS}")
    print("="*80 + "\n")
    
    start_time = datetime.now()
    
    # Executar otimização
    study.optimize(
        optuna_objective,
        n_trials=N_RANDOM_TRIALS,
        show_progress_bar=True,
        callbacks=[
            lambda study, trial: study.trials_dataframe().to_csv(
                'optuna_results_partial.csv', index=False
            ) if trial.number % 5 == 0 else None
        ]
    )
    
    end_time = datetime.now()
    duration = end_time - start_time
    
    print(f"\n{'='*80}")
    print(f"✅ Busca Optuna concluída!")
    print(f"⏱️ Tempo total: {duration}")
    print(f"🏆 Melhor RMSE: {study.best_value:.6f}")
    print(f"{'='*80}\n")
    
    # Extrair resultados
    results = [trial.user_attrs.get('full_result') for trial in study.trials 
               if 'full_result' in trial.user_attrs]
    
    # Salvar resultados
    df_results = pd.DataFrame(results)
    df_results.to_csv('optuna_results.csv', index=False)
    
    # Mostrar melhores parâmetros
    print("🏆 MELHORES HIPERPARÂMETROS (Optuna):")
    print("="*80)
    for key, value in study.best_params.items():
        print(f"{key:25s}: {value}")
    print("="*80)
    
    # Visualização Optuna
    try:
        from optuna.visualization import plot_optimization_history, plot_param_importances
        
        # Histórico de otimização
        fig1 = plot_optimization_history(study)
        fig1.write_image('optuna_history.png')
        fig1.show()
        
        # Importância dos parâmetros
        fig2 = plot_param_importances(study)
        fig2.write_image('optuna_importance.png')
        fig2.show()
        
        print("📊 Gráficos salvos: optuna_history.png, optuna_importance.png")
    except Exception as e:
        print(f"⚠️ Não foi possível gerar visualizações Optuna: {e}")
        print("   Instale: pip install plotly kaleido")
    
else:
    print("❌ Optuna não está instalado. Use: pip install optuna")
"""

print("💡 Código Optuna disponível acima (descomente para usar)")

💡 Código Optuna disponível acima (descomente para usar)


# 📊 Resumo: Como Acelerar a Busca

## ⚡ Configurações Atuais (edite a célula de configuração acima):

| Estratégia | Configuração | Impacto na Velocidade | Impacto na Qualidade |
|------------|-------------|----------------------|---------------------|
| **Random Search** | `USE_RANDOM_SEARCH=True` | 🚀🚀🚀 Muito Alto (50 trials vs 1536) | ⚠️ Médio (pode perder ótimo global) |
| **Epochs Reduzidas** | `SEARCH_EPOCHS=10` | 🚀🚀🚀 Muito Alto (5x mais rápido) | ✅ Baixo (retreina depois) |
| **Early Stopping Agressivo** | `SEARCH_PATIENCE=3` | 🚀🚀 Alto | ✅ Baixo |
| **Subset de Dados** | `USE_DATA_SUBSET=True` | 🚀🚀 Alto (2x mais rápido) | ⚠️ Médio |
| **Espaço Reduzido** | Menos valores por parâmetro | 🚀 Médio | ⚠️ Médio |
| **Optuna (Bayesiano)** | Descomente código Optuna | 🚀🚀🚀 Muito Alto (mais inteligente) | ✅✅ Melhor! |

## 🎯 Recomendações:

### Para Teste Rápido (~1-2 horas):
```python
USE_RANDOM_SEARCH = True
N_RANDOM_TRIALS = 20-30
SEARCH_EPOCHS = 5-10
USE_DATA_SUBSET = False  # Manter dados completos
```

### Para Busca Balanceada (~3-5 horas):
```python
USE_RANDOM_SEARCH = True
N_RANDOM_TRIALS = 50-100
SEARCH_EPOCHS = 10-15
USE_DATA_SUBSET = False
```

### Para Busca Completa e Precisa (~8-12 horas):
```python
USE_RANDOM_SEARCH = False  # Grid completo
SEARCH_EPOCHS = 20-30
USE_DATA_SUBSET = False
# OU usar Optuna com 100+ trials
```

## 💡 Dica Principal:
**Use poucas epochs na busca (5-10) e retreine o melhor modelo com 50+ epochs depois!**
Isso garante velocidade na exploração e qualidade no modelo final.

In [118]:
def run_experiment(
    features,
    context_length,
    forecast_horizon,
    d_model,
    num_attention_heads,
    num_hidden_layers,
    ffn_dim,
    dropout,
    patch_length,
    learning_rate,
    batch_size,
    experiment_id,
    use_fast_config=True
):
    """
    Executa um experimento completo com os parâmetros fornecidos.
    Retorna um dicionário com os resultados.
    
    use_fast_config: Se True, usa configurações otimizadas para busca rápida
    """
    print(f"\n{'='*80}")
    print(f"🧪 EXPERIMENTO {experiment_id}")
    print(f"{'='*80}")
    print(f"Features: {features}")
    print(f"Context Length: {context_length}")
    print(f"d_model: {d_model}, heads: {num_attention_heads}, layers: {num_hidden_layers}")
    print(f"LR: {learning_rate}, Batch: {batch_size}, Patch: {patch_length}")
    print(f"{'='*80}\n")
    
    try:
        # Escolher configuração (rápida ou completa)
        config_params = FIXED_PARAMS_FAST if use_fast_config else FIXED_PARAMS
        
        # 1. Preparar dados (patch já aplicado no repo.executar)
        tsp, train_ds, valid_ds, test_ds = repo.executar(
            timestamp_col=TIMESTAMP_COLUMN,
            train_frac=config_params['train_frac'],
            valid_frac=config_params['valid_frac'],
            context_length=context_length,
            features=features,
            target=TARGET_COLUMN,
            id_cols=ID_COLUMNS,
            forecast_horizon=forecast_horizon
        )
        
        # Garantir datasets writeable (evita "assignment destination is read-only")
        train_ds = make_df_writable(train_ds)
        valid_ds = make_df_writable(valid_ds)
        test_ds = make_df_writable(test_ds)
        
        # 2. Configurar modelo
        config = PatchTSTConfig(
            do_mask_input=False,
            context_length=context_length,
            patch_length=patch_length,
            num_input_channels=len(TARGET_COLUMN),
            patch_stride=patch_length,
            prediction_length=forecast_horizon,
            d_model=d_model,
            num_attention_heads=num_attention_heads,
            num_hidden_layers=num_hidden_layers,
            ffn_dim=ffn_dim,
            dropout=dropout,
            head_dropout=dropout,
            pooling_type=None,
            channel_attention=True,
            scaling='std',
            loss='mse',
            pre_norm=True,
            norm_type='batchnorm',
        )
        
        model = PatchTSTForPrediction(config=config).to(device).to(dtype)
        
        # 3. Configurar treinamento
        train_args = TrainingArguments(
            output_dir="./grid_search_temp",  # Pasta única para todos os experimentos
            overwrite_output_dir=True,
            learning_rate=learning_rate,
            num_train_epochs=config_params['epochs'],
            do_eval=True,
            eval_strategy="epoch",
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            dataloader_num_workers=config_params['num_workers'],
            save_strategy="no",  # Não salvar checkpoints durante busca
            logging_strategy="epoch",
            logging_dir=None,  # Sem logs individuais
            load_best_model_at_end=False,  # Não precisa carregar melhor modelo na busca
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            label_names=["future_values"],
            report_to="none",  # Desabilitar wandb/tensorboard
        )
        
        early_stopping = EarlyStoppingCallback(
            early_stopping_patience=config_params['early_stopping_patience'],
            early_stopping_threshold=0.001
        )
        
        trainer = Trainer(
            model=model,
            args=train_args,
            train_dataset=train_ds,
            eval_dataset=valid_ds,
            compute_metrics=compute_metrics,
            callbacks=[early_stopping]
        )
        
        # 4. Treinar
        train_result = trainer.train()
        
        # 5. Avaliar no conjunto de validação
        eval_result = trainer.evaluate()
        
        # 6. Coletar métricas
        result = {
            'experiment_id': experiment_id,
            'features': str(features),
            'num_features': len(features),
            'context_length': context_length,
            'forecast_horizon': forecast_horizon,
            'd_model': d_model,
            'num_attention_heads': num_attention_heads,
            'num_hidden_layers': num_hidden_layers,
            'ffn_dim': ffn_dim,
            'dropout': dropout,
            'patch_length': patch_length,
            'learning_rate': learning_rate,
            'batch_size': batch_size,
            'train_loss': train_result.training_loss,
            'eval_loss': eval_result['eval_loss'],
            'eval_MSE': eval_result.get('eval_MSE', None),
            'eval_MAE': eval_result.get('eval_MAE', None),
            'eval_RMSE': eval_result.get('eval_RMSE', None),
            'eval_MAPE': eval_result.get('eval_MAPE', None),
            'epochs_trained': train_result.global_step // len(train_ds) * batch_size,
            'status': 'success'
        }
        
        print(f"✅ Experimento {experiment_id} concluído com sucesso!")
        print(f"   Val Loss: {eval_result['eval_loss']:.6f} | RMSE: {result['eval_RMSE']:.6f}")
        
        # Limpar memória
        del model, trainer, train_ds, valid_ds, test_ds, tsp
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
        
        return result
        
    except Exception as e:
        print(f"❌ Erro no experimento {experiment_id}: {str(e)}")
        return {
            'experiment_id': experiment_id,
            'features': str(features),
            'context_length': context_length,
            'd_model': d_model,
            'learning_rate': learning_rate,
            'batch_size': batch_size,
            'status': 'failed',
            'error': str(e)
        }

print("✓ Função run_experiment definida (com modo rápido)")

✓ Função run_experiment definida (com modo rápido)


In [119]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning, message='.*pin_memory.*')

def make_dataset_writable(dataset):
    """
    Garante que todos os arrays no dataset sejam modificáveis (writable).
    Corrige o erro: 'assignment destination is read-only'
    """
    # Acessar o objeto interno do dataset
    if hasattr(dataset, 'dataset') and hasattr(dataset.dataset, 'df'):
        df = dataset.dataset.df
        
        # Converter todas as colunas numéricas para arrays modificáveis
        for col in df.columns:
            if df[col].dtype in [np.float32, np.float64, np.int32, np.int64]:
                arr = df[col].values
                if not arr.flags.writeable:
                    # Criar cópia modificável
                    df[col] = np.array(arr, copy=True)
    
    return dataset

print("✓ Função make_dataset_writable definida")
print("✓ Warnings de pin_memory desabilitados")

✓ Função make_dataset_writable definida
✓ Warnings de pin_memory desabilitados


In [120]:
# Função utilitária simples para garantir arrays writeable

def make_df_writable(dataset):
    """Garante que o DataFrame interno do dataset tenha arrays modificáveis."""
    if hasattr(dataset, 'dataset') and hasattr(dataset.dataset, 'df'):
        df = dataset.dataset.df
        for col in df.select_dtypes(include=[np.number]).columns:
            df[col] = df[col].to_numpy(copy=True)
    return dataset

print("✓ Função make_df_writable definida")

✓ Função make_df_writable definida


In [121]:
# Verificar se o patch já foi aplicado antes
if not hasattr(repo, '_original_executar'):
    # Salvar referência original
    repo._original_executar = repo.executar
    
    def executar_with_writable_fix(*args, **kwargs):
        """Wrapper que garante arrays modificáveis - versão sem recursão"""
        # Chamar método original salvo
        tsp, train_ds, valid_ds, test_ds = repo._original_executar(*args, **kwargs)
        
        # Aplicar correção nos datasets
        for ds in [train_ds, valid_ds, test_ds]:
            if hasattr(ds, 'dataset') and hasattr(ds.dataset, 'df'):
                df = ds.dataset.df
                # Forçar cópia dos dados numéricos que estão read-only
                for col in df.columns:
                    if df[col].dtype in [np.float32, np.float64, np.int32, np.int64]:
                        arr = df[col].values
                        if hasattr(arr, 'flags') and not arr.flags.writeable:
                            df[col] = arr.copy()
        
        return tsp, train_ds, valid_ds, test_ds
    
    # Aplicar o patch
    repo.executar = executar_with_writable_fix
    print("✅ Patch aplicado: Arrays agora são modificáveis (sem recursão)")
else:
    print("✓ Patch já estava aplicado anteriormente")

✅ Patch aplicado: Arrays agora são modificáveis (sem recursão)


# Executar Busca Otimizada (Random ou Grid)

⚡ **MODO RÁPIDO ATIVADO**: Configurações otimizadas para máxima velocidade
- Reduz epochs durante busca (5-10)
- Early stopping agressivo
- Random search ao invés de grid completo (opcional)
- **Não salva checkpoints** (save_strategy="no")
- Usa **uma única pasta temporária** para todos experimentos
- O melhor modelo será retreinado com configurações completas depois

💡 **Economia de Espaço**: Ao final, a pasta temporária será limpa automaticamente

In [122]:
# Gerar todas as combinações
all_combinations = list(product(
    FEATURE_COMBINATIONS_FAST,
    CONTEXT_LENGTHS_FAST,
    [1],  # FORECAST_HORIZONS fixo
    MODEL_PARAMS_FAST['d_model'],
    MODEL_PARAMS_FAST['num_attention_heads'],
    MODEL_PARAMS_FAST['num_hidden_layers'],
    MODEL_PARAMS_FAST['ffn_dim'],
    MODEL_PARAMS_FAST['dropout'],
    MODEL_PARAMS_FAST['patch_length'],
    TRAINING_PARAMS_FAST['learning_rate'],
    TRAINING_PARAMS_FAST['batch_size'],
))

print(f"🔍 Total de combinações possíveis: {len(all_combinations)}")

# Aplicar Random Search se ativado
if USE_RANDOM_SEARCH and len(all_combinations) > N_RANDOM_TRIALS:
    import random
    random.seed(RANDOM_STATE)
    selected_combinations = random.sample(all_combinations, N_RANDOM_TRIALS)
    print(f"⚡ Modo RANDOM SEARCH: Testando {N_RANDOM_TRIALS} combinações aleatórias")
else:
    selected_combinations = all_combinations
    print(f"📊 Modo GRID SEARCH: Testando TODAS as {len(all_combinations)} combinações")

print(f"⏱️ Tempo estimado (~1 min/exp): {len(selected_combinations) / 60:.1f} horas\n")

# Lista para armazenar resultados
results = []

# Executar busca
start_time = datetime.now()
print(f"🚀 Iniciando busca às {start_time.strftime('%H:%M:%S')}\n")

for i, combo in enumerate(selected_combinations, 1):
    features, context_length, forecast_horizon, d_model, num_heads, num_layers, ffn_dim, dropout, patch_length, lr, batch = combo
    
    result = run_experiment(
        features=features,
        context_length=context_length,
        forecast_horizon=forecast_horizon,
        d_model=d_model,
        num_attention_heads=num_heads,
        num_hidden_layers=num_layers,
        ffn_dim=ffn_dim,
        dropout=dropout,
        patch_length=patch_length,
        learning_rate=lr,
        batch_size=batch,
        experiment_id=i,
        use_fast_config=True  # Usar configuração rápida
    )
    
    results.append(result)
    
    # Salvar resultados parciais a cada 5 experimentos
    if i % 5 == 0:
        df_temp = pd.DataFrame(results)
        df_temp.to_csv('grid_search_results_partial.csv', index=False)
        elapsed = datetime.now() - start_time
        avg_time = elapsed.total_seconds() / i
        remaining = (len(selected_combinations) - i) * avg_time
        print(f"\n💾 Progresso: {i}/{len(selected_combinations)} ({i/len(selected_combinations)*100:.1f}%)")
        print(f"⏱️ Tempo decorrido: {elapsed} | Restante: ~{remaining/60:.0f} min\n")

end_time = datetime.now()
duration = end_time - start_time

print(f"\n{'='*80}")
print(f"✅ Busca concluída!")
print(f"⏱️ Tempo total: {duration}")
print(f"📊 Experimentos bem-sucedidos: {sum(1 for r in results if r['status'] == 'success')}/{len(results)}")
print(f"⚡ Tempo médio por experimento: {duration.total_seconds()/len(results):.1f}s")
print(f"{'='*80}\n")

🔍 Total de combinações possíveis: 1536
⚡ Modo RANDOM SEARCH: Testando 50 combinações aleatórias
⏱️ Tempo estimado (~1 min/exp): 0.8 horas

🚀 Iniciando busca às 19:13:12


🧪 EXPERIMENTO 1
Features: ['Vol_lag_1']
Context Length: 512
d_model: 64, heads: 8, layers: 2
LR: 0.0001, Batch: 64, Patch: 16

Carregando dados de C:\Users\Andre\OneDrive\Documentos\Github\iniciacao-cientifica\data\raw\BTCUSDT_5m.txt
Calculando features de volatilidade
Treino: 1278 amostras | Val: 256 | Teste: 512


Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


❌ Erro no experimento 1: assignment destination is read-only

🧪 EXPERIMENTO 2
Features: ['Vol_lag_1', 'Vol_week_mean', 'Vol_month_mean']
Context Length: 256
d_model: 128, heads: 16, layers: 3
LR: 0.0001, Batch: 32, Patch: 16

Carregando dados de C:\Users\Andre\OneDrive\Documentos\Github\iniciacao-cientifica\data\raw\BTCUSDT_5m.txt
Calculando features de volatilidade


KeyboardInterrupt: 

In [ ]:
# Salvar resultados finais
df_results = pd.DataFrame(results)
df_results.to_csv('grid_search_results.csv', index=False)
print("💾 Resultados salvos em 'grid_search_results.csv'")

# Filtrar apenas experimentos bem-sucedidos
df_success = df_results[df_results['status'] == 'success'].copy()

if len(df_success) > 0:
    # Ordenar por eval_loss (menor é melhor)
    df_success = df_success.sort_values('eval_loss')
    
    print("\n📊 TOP 10 MELHORES CONFIGURAÇÕES (por eval_loss):")
    print("="*120)
    
    display_cols = ['experiment_id', 'eval_loss', 'eval_RMSE', 'eval_MAE', 'eval_MAPE', 
                    'd_model', 'num_attention_heads', 'num_hidden_layers', 
                    'learning_rate', 'batch_size', 'context_length', 'num_features']
    
    print(df_success[display_cols].head(10).to_string(index=False))
    
    print("\n🏆 MELHOR CONFIGURAÇÃO:")
    print("="*120)
    best = df_success.iloc[0]
    for key in best.index:
        if key not in ['status', 'error']:
            print(f"{key:25s}: {best[key]}")
else:
    print("❌ Nenhum experimento foi bem-sucedido!")

💾 Resultados salvos em 'grid_search_results.csv'
❌ Nenhum experimento foi bem-sucedido!


In [ ]:
# Limpar pasta temporária para economizar espaço
import shutil
import os

temp_dir = "./grid_search_temp"
if os.path.exists(temp_dir):
    try:
        shutil.rmtree(temp_dir)
        print("🧹 Pasta temporária limpa com sucesso")
    except Exception as e:
        print(f"⚠️ Não foi possível limpar pasta temporária: {e}")
else:
    print("✓ Nenhuma pasta temporária para limpar")

✓ Nenhuma pasta temporária para limpar


# Análise Visual dos Resultados

In [ ]:
import seaborn as sns

if len(df_success) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('Análise de Hiperparâmetros - Grid Search', fontsize=16, y=1.02)
    
    # 1. Impacto de d_model
    sns.boxplot(data=df_success, x='d_model', y='eval_RMSE', ax=axes[0,0])
    axes[0,0].set_title('Impacto de d_model no RMSE')
    axes[0,0].set_ylabel('RMSE (Validation)')
    
    # 2. Impacto de num_attention_heads
    sns.boxplot(data=df_success, x='num_attention_heads', y='eval_RMSE', ax=axes[0,1])
    axes[0,1].set_title('Impacto de Attention Heads no RMSE')
    axes[0,1].set_ylabel('RMSE (Validation)')
    
    # 3. Impacto de num_hidden_layers
    sns.boxplot(data=df_success, x='num_hidden_layers', y='eval_RMSE', ax=axes[0,2])
    axes[0,2].set_title('Impacto de Hidden Layers no RMSE')
    axes[0,2].set_ylabel('RMSE (Validation)')
    
    # 4. Impacto de learning_rate
    sns.boxplot(data=df_success, x='learning_rate', y='eval_RMSE', ax=axes[1,0])
    axes[1,0].set_title('Impacto de Learning Rate no RMSE')
    axes[1,0].set_ylabel('RMSE (Validation)')
    axes[1,0].tick_params(axis='x', rotation=45)
    
    # 5. Impacto de context_length
    sns.boxplot(data=df_success, x='context_length', y='eval_RMSE', ax=axes[1,1])
    axes[1,1].set_title('Impacto de Context Length no RMSE')
    axes[1,1].set_ylabel('RMSE (Validation)')
    
    # 6. Impacto do número de features
    sns.boxplot(data=df_success, x='num_features', y='eval_RMSE', ax=axes[1,2])
    axes[1,2].set_title('Impacto do Número de Features no RMSE')
    axes[1,2].set_ylabel('RMSE (Validation)')
    axes[1,2].set_xlabel('Número de Features')
    
    plt.tight_layout()
    plt.savefig('grid_search_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("📊 Gráficos salvos em 'grid_search_analysis.png'")
else:
    print("❌ Sem dados para visualização")

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
# Análise de correlação entre hiperparâmetros e performance
if len(df_success) > 0:
    numeric_cols = ['d_model', 'num_attention_heads', 'num_hidden_layers', 'ffn_dim', 
                    'dropout', 'patch_length', 'learning_rate', 'batch_size', 
                    'context_length', 'num_features', 'eval_RMSE', 'eval_MAE', 'eval_MAPE']
    
    corr_matrix = df_success[numeric_cols].corr()
    
    plt.figure(figsize=(14, 10))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                square=True, linewidths=1, cbar_kws={"shrink": 0.8})
    plt.title('Matriz de Correlação - Hiperparâmetros vs Performance', fontsize=14, pad=20)
    plt.tight_layout()
    plt.savefig('correlation_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n📊 Correlações mais fortes com RMSE:")
    rmse_corr = corr_matrix['eval_RMSE'].abs().sort_values(ascending=False)
    print(rmse_corr.head(10))
else:
    print("❌ Sem dados para análise de correlação")

# Testar a Melhor Configuração no Conjunto de Teste

In [ ]:
if len(df_success) > 0:
    # Obter a melhor configuração
    best = df_success.iloc[0]
    
    print("🏆 Retreinando modelo com a MELHOR CONFIGURAÇÃO encontrada:")
    print("="*80)
    print(f"Features: {best['features']}")
    print(f"Context Length: {best['context_length']}")
    print(f"d_model: {best['d_model']}, heads: {best['num_attention_heads']}, layers: {best['num_hidden_layers']}")
    print(f"FFN dim: {best['ffn_dim']}, dropout: {best['dropout']}, patch: {best['patch_length']}")
    print(f"LR: {best['learning_rate']}, Batch: {best['batch_size']}")
    print(f"Validation RMSE (busca rápida): {best['eval_RMSE']:.6f}")
    print("="*80)
    print(f"\n⚠️ RETREINANDO COM CONFIGURAÇÕES COMPLETAS:")
    print(f"   - Epochs: {SEARCH_EPOCHS} → {FINAL_EPOCHS}")
    print(f"   - Early Stopping Patience: {SEARCH_PATIENCE} → {FINAL_PATIENCE}")
    print(f"   - Dados: 100% (sem subset)")
    print("="*80 + "\n")
    
    # Converter string de features de volta para lista
    import ast
    best_features = ast.literal_eval(best['features'])
    
    # Preparar dados com a melhor configuração (DADOS COMPLETOS)
    tsp_best, train_ds_best, valid_ds_best, test_ds_best = repo.executar(
        timestamp_col=TIMESTAMP_COLUMN,
        train_frac=0.7,  # Usar dados completos
        valid_frac=0.1,  # Usar dados completos
        context_length=int(best['context_length']),
        features=best_features,
        target=TARGET_COLUMN,
        id_cols=ID_COLUMNS,
        forecast_horizon=int(best['forecast_horizon'])
    )
    
    # Criar modelo com melhor configuração
    best_config = PatchTSTConfig(
        do_mask_input=False,
        context_length=int(best['context_length']),
        patch_length=int(best['patch_length']),
        num_input_channels=len(TARGET_COLUMN),
        patch_stride=int(best['patch_length']),
        prediction_length=int(best['forecast_horizon']),
        d_model=int(best['d_model']),
        num_attention_heads=int(best['num_attention_heads']),
        num_hidden_layers=int(best['num_hidden_layers']),
        ffn_dim=int(best['ffn_dim']),
        dropout=float(best['dropout']),
        head_dropout=float(best['dropout']),
        pooling_type=None,
        channel_attention=True,
        scaling='std',
        loss='mse',
        pre_norm=True,
        norm_type='batchnorm',
    )
    
    best_model = PatchTSTForPrediction(best_config).to(device).to(dtype)
    
    # Treinar modelo final com CONFIGURAÇÕES COMPLETAS
    final_train_args = TrainingArguments(
        output_dir="./best_model_final",
        overwrite_output_dir=True,
        learning_rate=float(best['learning_rate']),
        num_train_epochs=FINAL_EPOCHS,  # Mais epochs!
        do_eval=True,
        eval_strategy="epoch",
        per_device_train_batch_size=int(best['batch_size']),
        per_device_eval_batch_size=int(best['batch_size']),
        dataloader_num_workers=0,
        save_strategy="epoch",
        logging_strategy="epoch",
        save_total_limit=1,
        fp16=(dtype == torch.float16),
        bf16=(dtype == torch.bfloat16),
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        label_names=["future_values"]
    )
    
    final_trainer = Trainer(
        model=best_model,
        args=final_train_args,
        train_dataset=train_ds_best,
        eval_dataset=valid_ds_best,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=FINAL_PATIENCE)]
    )
    
    print("\n🚀 Treinando modelo final com configurações completas...")
    final_trainer.train()
    
    print("\n📊 Avaliando no conjunto de TESTE...")
else:
    print("❌ Não há configuração vencedora para testar")

In [ ]:
# Avaliar e visualizar no conjunto de teste
if len(df_success) > 0:
    test_df = repo.carregar_df()
    forecast_dates, test_metrics = evaluate_and_visualize(
        model=best_model,
        test_dataset=test_ds_best,
        tsp=tsp_best,
        test_df=test_df,
        model_name="PatchTST - Melhor Configuração"
    )
    
    print("\n" + "="*80)
    print("🎯 RESULTADOS FINAIS NO CONJUNTO DE TESTE")
    print("="*80)
    print(f"MSE:  {test_metrics['MSE']:.6e}")
    print(f"MAE:  {test_metrics['MAE']:.6e}")
    print(f"RMSE: {test_metrics['RMSE']:.6e}")
    print(f"MAPE: {test_metrics['MAPE']:.2f}%")
    print("="*80)
    
    # Salvar modelo final
    best_model.save_pretrained("./best_model_patchtst")
    print("\n💾 Modelo final salvo em './best_model_patchtst'")
else:
    print("❌ Não há modelo para avaliar")

# 🔧 Preparação dos Dados

⚠️ **IMPORTANTE**: Execute as próximas 3 células antes de rodar o grid search:
1. Carregar o repositório de dados
2. Aplicar correção para arrays read-only
3. Desabilitar warnings de pin_memory

Isso corrige o erro: `assignment destination is read-only`

In [ ]:
tsp, train_ds, valid_ds, test_ds = repo.executar(
    timestamp_col=TIMESTAMP_COLUMN,
    train_frac=TRAIN_FRAC,
    valid_frac=VALID_FRAC,
    context_length=CONTEXT_LENGTH,
    features=FEATURES,
    target=TARGET_COLUMN,
    id_cols=ID_COLUMNS,
    forecast_horizon=FORECAST_HORIZON
)

Carregando dados de C:\Users\Andre\OneDrive\Documentos\Github\iniciacao-cientifica\data\raw\BTCUSDT_5m.txt
Calculando features de volatilidade
Treino: 1278 amostras | Val: 256 | Teste: 512


In [ ]:
raise

In [ ]:
from transformers import (
    EarlyStoppingCallback,
    PatchTSTConfig,
    PatchTSTForPrediction,
    Trainer,
    TrainingArguments    
)

In [ ]:
config = PatchTSTConfig(
    do_mask_input=False,
    context_length=CONTEXT_LENGTH,
    patch_length=PATCH_LENGTH,
    num_input_channels=len(TARGET_COLUMN),
    patch_stride=PATCH_LENGTH,
    prediction_length=FORECAST_HORIZON,
    d_model=128,
    num_attention_heads=16,
    num_hidden_layers=3,
    ffn_dim=512,
    dropout=0.2,
    head_dropout=0.2,
    pooling_type=None,
    channel_attention=True, # Ativação da atenção entre canais
    scaling='std',
    loss='mse',
    pre_norm=True,
    norm_type='batchnorm',
)

model = PatchTSTForPrediction(
    config=config
    ).to(device).to(dtype)

In [ ]:
train_args = TrainingArguments(
    output_dir="./patchtst_volatility",
    overwrite_output_dir=True,
    learning_rate=LEARNING_RATE,
    num_train_epochs=EPOCHS,
    do_eval=True,
    eval_strategy="epoch",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    dataloader_num_workers=NUM_WORKERS,
    save_strategy="epoch",
    logging_strategy="epoch",
    save_total_limit=1,
    fp16=(dtype == torch.float16),
    bf16=(dtype == torch.bfloat16),
    logging_dir="./logs_patchtst_volatility",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    label_names=["future_values"]
)

early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience=5,
    early_stopping_threshold=0.001
)

trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    compute_metrics=compute_metrics,
    callbacks=[early_stopping_callback]
)

In [ ]:
print("Iniciando o treinamento do PatchTST para volatilidade...")
trainer.train()

Iniciando o treinamento do PatchTST para volatilidade...


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 